# SMT and LP on three small problems 

**998MG Safe and Verified AI — 2026/27 · Università degli Studi di Trieste**

The three programs shown in class, run with an SMT solver (Z3). The last two are also solved
with linear programming (SciPy, which calls the HiGHS solver).

| problem | SMT (Z3) | LP / MILP |
|---|---|---|
| **1. a simple formula** | yes | — |
| **2. exam scheduling** | yes, in pure Booleans | a 0/1 integer program (MILP) |
| **3. the robot** | yes | one LP, which also gives the range of every variable |

The SMT-LIB2 programs are the files in `data/`. The notebook runs them as they are, so what you
see is exactly the code of the slides. Everything runs in a second.

In [ ]:
# Colab only (locally, requirements.txt has already installed it): uncomment the next line
# %pip install -q z3-solver

In [ ]:
# SciPy for LP and MILP, Z3 for SMT
import urllib.request
from pathlib import Path

import numpy as np
from scipy.optimize import linprog, milp, LinearConstraint, Bounds
import z3

## A helper

`run_smt2` runs an SMT-LIB2 file with Z3, exactly as `z3 file.smt2` on the command line, and
prints first the program and then Z3's answer. It reads the file from `data/`; on Colab, where
the folder is not there, it downloads it from the course repository.

In [ ]:
DATA = Path("data")
URL  = "https://raw.githubusercontent.com/lauranenzi/svai-lab/main/lab-reluplex-milp/data/"

def run_smt2(name, show=True):
    path = DATA / name
    # locally the file is in data/; on Colab that folder does not exist, so we download it from the course repo
    text = path.read_text() if path.exists() else urllib.request.urlopen(URL + name).read().decode()
    if show:
        print(text)
        print("----------------  Z3 output  ----------------")
    # a fresh context per file, so that the declarations of one file do not clash with the next
    ctx = z3.Context()
    print(z3.Z3_eval_smtlib2_string(ctx.ref(), text))

## 1. A simple formula

$$(x > 1 \;\lor\; y < 2) \;\land\; (x + y \le 4) \;\land\; (\neg(y < 2) \;\lor\; x \le 3), \qquad x, y \in \mathbb{R}$$

### With SMT

Linear real arithmetic (`QF_LRA`): Z3 handles the $\lor$, the $\neg$ and the strict inequalities on its own.

In [ ]:
run_smt2("exercise1.smt2")

Z3 returns **a** model, here $x = 0,\ y = 0$: $y < 2$ makes the first clause true, $x \le 3$ the
third one.

## 2. Exam scheduling

Six exams $E_1,\dots,E_6$, four slots $S_1,\dots,S_4$ ($S_1, S_2$ morning, $S_3, S_4$ afternoon).

- every exam goes in **exactly one** slot;
- **conflicts** (common students): $E_1$–$E_2$, $E_3$–$E_4$, $E_3$–$E_6$ never in the same slot;
- **prerequisites**: $E_1$ in the morning, $E_4$ in the afternoon.

One unknown per pair: $x_{ij}$ = "exam $E_i$ is in slot $S_j$", 24 unknowns in total.

### With SMT, in pure Booleans

The program checks two things: whether a timetable exists (the first `check-sat`, then the slot
of each exam), and, inside `push`/`pop`, whether the timetable of the slide is a model.

In [ ]:
run_smt2("exam_slots.smt2")

### With MILP: the same problem as a 0/1 integer program

Each Boolean $x_{ij}$ becomes a variable in $\{0,1\}$ and each constraint becomes a linear one:

| logic | linear |
|---|---|
| exactly one of $x_{i1},\dots,x_{i4}$ | $x_{i1} + x_{i2} + x_{i3} + x_{i4} = 1$ |
| $\neg x_{ij} \lor \neg x_{kj}$ (conflict) | $x_{ij} + x_{kj} \le 1$ |
| $x_{11} \lor x_{12}$ (prerequisite) | $x_{11} + x_{12} \ge 1$ |

One equality replaces the "at least one" clause plus the six "at most one" clauses of the SMT
version. There is no objective: we only want a feasible point, so $c = 0$.

We build it with loops instead of writing all 24 variables by hand.

In [ ]:
EXAMS, SLOTS = 6, 4
CONFLICTS = [(1, 2), (3, 4), (3, 6)]
MORNING, AFTERNOON = [1, 2], [3, 4]

# column of x_ij in the vector of 24 unknowns (exams and slots numbered from 1, as on the slide)
def col(i, j):
    return SLOTS * (i - 1) + (j - 1)

rows, lo, hi = [], [], []
# one row:  lo <= sum of the x_ij in `pairs` <= hi
def add_row(pairs, l, h):
    r = np.zeros(EXAMS * SLOTS)
    for i, j in pairs:
        r[col(i, j)] = 1
    rows.append(r); lo.append(l); hi.append(h)

for i in range(1, EXAMS + 1):                         # 1) each exam exactly once
    add_row([(i, j) for j in range(1, SLOTS + 1)], 1, 1)
for i, k in CONFLICTS:                                # 2) conflicts: x_ij + x_kj <= 1, for every slot j
    for j in range(1, SLOTS + 1):
        add_row([(i, j), (k, j)], -np.inf, 1)
add_row([(1, j) for j in MORNING], 1, np.inf)         # 3) E1 in the morning
add_row([(4, j) for j in AFTERNOON], 1, np.inf)       #    E4 in the afternoon

timetable_constraints = LinearConstraint(np.array(rows), lo, hi)
print(f"{len(rows)} constraints on {EXAMS * SLOTS} unknowns")

In [ ]:
# solve; `fixed` forces some x_ij = 1, `relax` replaces {0,1} by [0,1] (the LP relaxation)
def schedule(fixed=(), relax=False):
    lower = np.zeros(EXAMS * SLOTS)
    for i, j in fixed:
        lower[col(i, j)] = 1
    res = milp(np.zeros(EXAMS * SLOTS),                       # no objective
               constraints=timetable_constraints,
               bounds=Bounds(lower, 1),
               integrality=0 if relax else 1)
    return None if res.x is None else np.round(res.x, 9).reshape(EXAMS, SLOTS) + 0.0

x = schedule()
for i in range(EXAMS):
    print(f"E{i + 1} -> S{int(np.argmax(x[i])) + 1}")

A timetable, possibly different from Z3's: both are feasible, and neither solver promises a
particular one. Now the timetable of the slide, $E_1, E_3, E_5$ in $S_1$, $E_2, E_6$ in $S_2$,
$E_4$ in $S_3$: we fix those six unknowns to 1 and ask again.

In [ ]:
SLIDE = [(1, 1), (3, 1), (5, 1), (2, 2), (6, 2), (4, 3)]
print("slide timetable is feasible:", schedule(fixed=SLIDE) is not None)

# and one that breaks a conflict: E3 and E4 together in S3
print("E3 and E4 both in S3 is feasible:", schedule(fixed=[(3, 3), (4, 3)]) is not None)

### And the LP relaxation?

Drop the integrality: $x_{ij} \in [0,1]$. The problem is now an LP, but its solution need not be
a timetable.

In [ ]:
x_lp = schedule(relax=True)
print("LP relaxation (row = exam, column = slot):")
print(x_lp)
print("integral:", bool(np.all((x_lp == 0) | (x_lp == 1))))

Here the LP happens to return an integral point: the simplex ends on a vertex of the polytope, and
this vertex is a timetable. Nothing guarantees it. The relaxation also contains fractional points,
for example the **average of two timetables**: it satisfies every linear constraint, but
"exam $E_i$ is half in $S_1$ and half in $S_2$" means nothing.

In [ ]:
# the slide timetable as a 6 x 4 matrix of 0/1
x_slide = np.zeros((EXAMS, SLOTS))
for i, j in SLIDE:
    x_slide[i - 1, j - 1] = 1

half = (x + x_slide) / 2                     # the midpoint of two timetables
A = timetable_constraints.A
print(half)
print("satisfies all the linear constraints:", bool(np.all((A @ half.ravel() >= np.array(lo) - 1e-9) &
                                                          (A @ half.ravel() <= np.array(hi) + 1e-9))))

For a **feasibility** question this is the important asymmetry:

- relaxation **infeasible** ⟹ the integer problem is infeasible too (no timetable exists);
- relaxation **feasible** ⟹ nothing, unless its point happens to be integral.

It is the same situation as the LP relaxation of the ReLU: a valid over-approximation, sometimes
exact, sometimes with points that the network never produces.

## 3. The robot

A robot at position $x_0$ moves for 2 time units at speed $v \in [0, 5]$ and reaches
$x_1 = x_0 + 2v$. It never goes below 0, starts at most at 10 and must end at least at 10.

### With SMT

In [ ]:
run_smt2("robot.smt2")

### With LP

The robot has only linear constraints, and they must all hold: this is exactly an LP. The bounds
of the single unknowns go in `bounds`, the equation $x_1 = x_0 + 2v$ in `A_eq`. There is no
objective: any feasible point will do.

In [ ]:
# unknowns (x0, x1, v)
A_eq, b_eq = [[-1, 1, -2]], [0]      # x1 = x0 + 2·v   <=>   -x0 + x1 - 2·v = 0
bounds = [(0, 10),                   # 0 <= x0 <= 10
          (10, None),                # x1 >= 0 and x1 >= 10
          (0, 5)]                    # 0 <= v <= 5

res = linprog(np.zeros(3), A_eq=A_eq, b_eq=b_eq, bounds=bounds)
print("feasible" if res.status == 0 else "infeasible", dict(zip(["x0", "x1", "v"], res.x.tolist())))

### What LP adds: the range of every variable

An LP can also **optimise**. Minimising and maximising each variable over the feasible set gives
its exact range: $2 \times 3 = 6$ LPs. This is the same computation as the bounds of the neurons
in verification, with a polytope instead of a box.

In [ ]:
for k, name in enumerate(["x0", "x1", "v"]):
    e = np.eye(3)[k]                                        # objective: the k-th unknown
    lp_min = linprog( e, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
    lp_max = linprog(-e, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
    print(f"{name} in [{lp_min.fun + 0.0:g}, {-lp_max.fun + 0.0:g}]")

So the robot ends somewhere in $x_1 \in [10, 20]$. Interval arithmetic on the same constraints
would only give $x_1 = x_0 + 2v \in [0, 20]$, because it ignores $x_1 \ge 10$ and the link between
$x_0$ and $v$. Z3 can answer the same question with `z3.Optimize`, as in the other notebook.

## Summary

| | SMT (Z3) | LP | MILP |
|---|---|---|---|
| linear constraints, all to hold | yes | **yes** | yes |
| $\lor$, $\neg$, Booleans | yes, natively | no | yes, with 0/1 variables |
| optimise an objective | `z3.Optimize` (extension) | **yes**, natively | yes |
| answer | `sat` + **a** model, or `unsat` | feasible + optimum, or infeasible | feasible + optimum, or infeasible |

